In [1]:
import numpy as np
from nltk.corpus import gutenberg

text = gutenberg.raw('shakespeare-macbeth.txt')  
corpus = text.split()

vocab = list(set(corpus))
w2i = {w:i for i,w in enumerate(vocab)}
i2w = {i:w for i,w in enumerate(vocab)}

d = 10  # embedding dimension, your choice
E = np.random.randn(len(vocab), d)  # fresh, random embedding matrix

# fixed-window 

In [2]:
# et: Concatenated embeddings
# getting the embeddings for my words to feed into my nueral net
def build_window_input(words, w2i, E):
    indices = [w2i[w] for w in words]
    vectors = E[indices]
    e_t =vectors.flatten()
    return e_t

words = corpus[:5]
window = build_window_input(words, w2i, E)
print(window.shape)

# ht: hidden representation 
# building ony forward pass of my nn for rnn
np.random.seed(42)
# i want 16 neurons in my hidden layer
W = np.random.randn(16, len(window))
b_h = np.random.randn(16)

def hidden_layer(e_t, W, b_h):
    return np.tanh(W @ e_t + b_h)

# y^_hat: Next-token distribution 
from scipy.special import softmax
U = np.random.randn(len(vocab), 16)   # (|V|, hidden_size)
b_o = np.random.randn(len(vocab))     # (|V|,)


def output_layer(h_t, U, b_o):
    return softmax(U @ h_t +b_o)

(50,)


In [3]:

# 1 FORWARD PASS
# Pass concatenated embeddings through hidden layer
h_t = hidden_layer(window, W, b_h)

# Pass hidden representation through output layer
y_hat = output_layer(h_t, U, b_o)

# checking the model shape
print("Input words:", words)

print("\nShapes:")
print("window e_t:", window.shape)
print("hidden h_t:", h_t.shape)
print("output y_hat:", y_hat.shape)

print("\nProbability check:")
print("Sum of probabilities:", y_hat.sum())

# look at top 5 probs after one forward pass
top_5_indices = np.argsort(y_hat)[-5:][::-1]
print("\nTop 5 predicted next words:")

for idx in top_5_indices:
    print(i2w[idx], y_hat[idx])

Input words: ['[The', 'Tragedie', 'of', 'Macbeth', 'by']

Shapes:
window e_t: (50,)
hidden h_t: (16,)
output y_hat: (5400,)

Probability check:
Sum of probabilities: 1.0

Top 5 predicted next words:
whin'd 0.20922195507279587
horrible 0.10348462911808941
Exit 0.06772454174370236
stooles. 0.05290424554078983
due 0.046543445219578775


# Vanilla RNN

In [4]:
np.random.seed(42)
# embedding dimension
d = 10

# hidden state size: number of neurons / values in the RNN memory
m = 16

# weights for the CURRENT word embedding x_t
W_x = np.random.randn(m, d)

# weights for the PREVIOUS hidden state h_(t-1)
W_h = np.random.randn(m ,m)

# bias for calculating the new hidden state
b_h = np.random.randn(m)

# Weights for my output layer
W_o = np.random.randn(len(w2i), m)

# bias for every output word
b_o = np.random.randn(len(w2i))

def rnn_step(x_t, h_prev, W_x, W_h, b_h):
    h_t = np.tanh(W_h @ h_prev + W_x @ x_t + b_h)
    return h_t

# output layer
def output_layer(h_t, W_o, b_o):
    y_hat = softmax(W_o @ h_t + b_o)
    return y_hat

# calculating the loss 
def nll_loss(y_hat, next_word):
    idx = w2i[next_word]
    next_word_prob = y_hat[idx]
    return -np.log(next_word_prob)

# gradient for softmax output = y hat - yt
def output_gradient(y_hat, next_word):
    one_hot = np.zeros(len(y_hat))
    idx = w2i[next_word]
    one_hot[idx] = 1
    return y_hat - one_hot

# gradient for W_o
def output_weight_gradient(y_hat, next_word, h_t):
    output_grad = output_gradient(y_hat, next_word)
    return np.outer(output_grad, h_t)

# gradient for my ht 
def hidden_output_gradient(W_o, output_grad):
    return np.transpose(W_o.T @ output_grad)

# gradient for tanh
def tanh_gradient(h_t, ht_grad):
    return np.multiply(ht_grad, (1-h_t**2))

# gradient for W_h
def W_h_gradient(h_prev, grad_tanh):
    return np.outer(grad_tanh, h_prev)

# gradient for W_x
def W_x_gradient(x_t, grad_tanh):
    return np.outer(grad_tanh, x_t)

# gradient for h(t-1):
def prev_h_gradient(W_h, grad_tanh):
    return grad_tanh @ W_h

def run_rnn(words):
    # handles batch_size = 1 
    # initial hidden state t=0
    h_0 = np.zeros(m)
    h_prev = h_0
    total_loss = 0 
    y_hats = []
    hidden_states = [h_0]
    for t in range(len(words)-1):
        next_word = words[t+1]
        x_t = E[w2i[words[t]]]
        h_t = rnn_step(x_t, h_prev, W_x, W_h, b_h)
        y_hat = output_layer(h_t, W_o, b_o)

        # store FULL probability distribution
        y_hats.append(y_hat)

        # keep track of avg loss
        loss = nll_loss(y_hat, next_word)
        total_loss += loss

        # store hidden state
        hidden_states.append(h_t)

        # recurrent step where my hidden layer at ht uses h_(t-1)
        h_prev = h_t
    avg_loss = total_loss/(len(words)-1)
    # only returns the y_hat for the last word, the in between y_hat gets replaced
    return y_hats, avg_loss, hidden_states


words = corpus[:1000]
y_hats, avg_loss, hidden_states = run_rnn(words)

print(len(y_hats))
print(y_hats[0].shape)
print(len(hidden_states))

999
(5400,)
1000


## One time Back Propogation Through Time

In [5]:
# initialize accumulators to store gradients across all time steps
grad_W_h = np.zeros((m, m))
grad_W_x = np.zeros((m, d))
grad_W_o = np.zeros((len(w2i), m))

# no future hidden-state error at the final time step
grad_h_future = np.zeros(m)

# get the final prediction, its true target, and the hidden state that produced it
final_y_hat = y_hats[len(words) - 2]
true_next_word = words[len(words) - 1]
h_t = hidden_states[len(words) - 1] # accounted for h0 at the start

# calculate output error: predicted probabilities - true one-hot vector
output_grad = output_gradient(final_y_hat, true_next_word)

# calculate how the final prediction contributes to the gradient of W_o
grad_output_weights = output_weight_gradient(final_y_hat, true_next_word, h_t)

# propagate the output error backwards into the hidden state h_t
grad_h_own = hidden_output_gradient(W_o, output_grad)

# combine this time step's own error with error coming backwards from future states
grad_h_t = grad_h_own + grad_h_future

# propagate the hidden-state error backwards through tanh
grad_tanh = tanh_gradient(h_t, grad_h_t)

x_t_emb = E[w2i[words[3]]]
grad_W_x_next = W_x_gradient(x_t_emb, grad_tanh)

h_prev = hidden_states[len(words) -2]
grad_W_h_next = W_h_gradient(h_prev, grad_tanh)

# pass the error backwards through W_h to the previous hidden state
grad_h_prev = prev_h_gradient(W_h, grad_tanh)

# this becomes the future error when we move to the previous time step
grad_h_future = grad_h_prev

## Through the history or words

In [6]:
# BPTT: calculate and accumulate gradients backwards through all time steps
grad_W_h = np.zeros((m, m))
grad_W_x = np.zeros((m, d))
grad_W_o = np.zeros((len(w2i), m))
grad_h_future = np.zeros(m)
grad_b_o = np.zeros(len(w2i))
grad_b_h = np.zeros(m)

for t in reversed(range(len(words) - 1)):

    # get the values needed for the current time step
    y_hat = y_hats[t]
    next_word = words[t + 1]
    h_t = hidden_states[t + 1]
    h_prev = hidden_states[t]
    x_t = E[w2i[words[t]]]

    # calculate the output error against the true next word
    output_grad = output_gradient(y_hat, next_word)

    # accumulate the gradient for the output weights
    grad_W_o += output_weight_gradient(y_hat, next_word, h_t)

    # propagate the output error backwards into the current hidden state
    grad_h_own = hidden_output_gradient(W_o, output_grad)

    # combine the current prediction error with error coming from future states
    grad_h_t = grad_h_own + grad_h_future

    # propagate the hidden-state error backwards through tanh
    grad_tanh = tanh_gradient(h_t, grad_h_t)

    # accumulate the bias gradients across time
    grad_b_o += output_grad
    grad_b_h += grad_tanh
    
    # accumulate the gradients for the recurrent and input weights
    grad_W_h += W_h_gradient(h_prev, grad_tanh)
    grad_W_x += W_x_gradient(x_t, grad_tanh)

    # pass the hidden-state error backwards to the previous time step
    grad_h_future = prev_h_gradient(W_h, grad_tanh)

## Training: updating the weights, bias, embedding for my vocab through gradient descent

In [7]:
# train the RNN repeatedly by running forward pass, BPTT, and parameter updates
lr = 0.001
epochs = 100

for epoch in range(epochs):

    # forward pass using the CURRENT parameters
    y_hats, avg_loss, hidden_states = run_rnn(words)

    # reset gradients before every backward pass
    grad_W_h = np.zeros_like(W_h)
    grad_W_x = np.zeros_like(W_x)
    grad_W_o = np.zeros_like(W_o)
    grad_b_h = np.zeros_like(b_h)
    grad_b_o = np.zeros_like(b_o)
    grad_E = np.zeros_like(E)

    grad_h_future = np.zeros(m)

    # BPTT through the sequence
    for t in reversed(range(len(words) - 1)):

        y_hat = y_hats[t]
        current_word = words[t]
        next_word = words[t + 1]

        h_t = hidden_states[t + 1]
        h_prev = hidden_states[t]
        x_t = E[w2i[current_word]]

        output_grad = output_gradient(y_hat, next_word)

        grad_W_o += output_weight_gradient(y_hat, next_word, h_t)
        
        grad_b_o += output_grad

        grad_h_own = hidden_output_gradient(W_o, output_grad)

        grad_h_t = grad_h_own + grad_h_future

        grad_tanh = tanh_gradient(h_t, grad_h_t)

        grad_W_h += W_h_gradient(h_prev, grad_tanh)

        grad_W_x += W_x_gradient(x_t, grad_tanh)

        grad_b_h += grad_tanh

        # gradient for the current word's embedding
        grad_x_t = W_x.T @ grad_tanh
        grad_E[w2i[current_word]] += grad_x_t

        # send error backwards through time
        grad_h_future = prev_h_gradient(W_h, grad_tanh)

    # update all parameters after BPTT
    W_o -= lr * grad_W_o
    W_h -= lr * grad_W_h
    W_x -= lr * grad_W_x

    b_o -= lr * grad_b_o
    b_h -= lr * grad_b_h

    E -= lr * grad_E

    # monitor training
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss = {avg_loss}")

Epoch 0: Loss = 15.305295075910296
Epoch 10: Loss = 15.57066704543929
Epoch 20: Loss = 15.281558224929293
Epoch 30: Loss = 14.812649259515332
Epoch 40: Loss = 14.357904200884649
Epoch 50: Loss = 14.041362125653935
Epoch 60: Loss = 13.686390907567969
Epoch 70: Loss = 13.364899350637799
Epoch 80: Loss = 13.155967617346336
Epoch 90: Loss = 12.830283921502202


## Chunks and Bacthes

In [8]:
def create_sequences(corpus, seq_len):
    inputs = []
    outputs = []
    for i in range(len(corpus) -seq_len): 
        inputs.append(corpus[i:i+seq_len])
        outputs.append(corpus[i+1:i+seq_len+1])
    return inputs, outputs

def create_batches(inputs, outputs, batch_size):
    input_batches = []
    output_batches = []
    for i in range(0, len(inputs), batch_size):
        input_batches.append(inputs[i:i+batch_size])
        output_batches.append(outputs[i:i+batch_size])
    return input_batches, output_batches

corpus_small = corpus[:100]

seq_len = 4
inputs, outputs = create_sequences(corpus_small, seq_len)

batch_size = 3
input_batches, output_batches = create_batches(inputs, outputs, batch_size)

print(len(input_batches))
print(len(input_batches[0]))
print(len(input_batches[0][0]))

print(input_batches[0])
print(output_batches[0])

32
3
4
[['[The', 'Tragedie', 'of', 'Macbeth'], ['Tragedie', 'of', 'Macbeth', 'by'], ['of', 'Macbeth', 'by', 'William']]
[['Tragedie', 'of', 'Macbeth', 'by'], ['of', 'Macbeth', 'by', 'William'], ['Macbeth', 'by', 'William', 'Shakespeare']]


## Temperature

In [9]:
def temperature_distribution(y_hat, temperature):
    total_prob = 0
    new_prob_distribution = []
    for prob in y_hat:
        prob = prob ** (1/temperature)
        new_prob_distribution.append(prob)
        total_prob += prob
    # convert list to numpy array
    new_prob_distribution = np.array(new_prob_distribution)
    new_prob_distribution = new_prob_distribution / total_prob 
    return new_prob_distribution

new_probs = temperature_distribution(y_hat, 0.5)

print(new_probs.sum())

1.0000000000000047


## Generation of text from RNN weight my updated weights from training

In [10]:
def generate_text(start_word, num_words , temperature, E, w2i, i2w, W_x, W_h, b_h,W_o, b_o):
    # stsart from previous memory
    h_prev = np.zeros(m)

    #first word provided by user
    current_word = start_word
    generated_words = [start_word]

    for _ in range(num_words):
        # get embedding for current word
        x_t = E[w2i[current_word]]

        # update hidden state
        h_t = rnn_step(x_t, h_prev, W_x, W_h, b_h)

        # predict probability distribution for next word
        y_hat = output_layer(h_t, W_o, b_o)

        # choose highest probability
        #highest_prob_idx = np.argmax(y_hat)
        #predicted_word = i2w[highest_prob_idx]

        #Temperature + argmax → still always chooses #1
        # Temperature + random.choice → samples according to the reshaped distribution ✓

        # using temperature instead of argmax
        new_prob_distribution = temperature_distribution(y_hat, temperature)
        # sample a vocabulary INDEX according to the adjusted probabilities
        sampled_idx = np.random.choice(len(new_prob_distribution), p=new_prob_distribution)
        predicted_word = i2w[sampled_idx]  

        #store predicted word
        generated_words.append(predicted_word)

        #predicted word becomes the next input
        current_word = predicted_word

        # current hidden state becomes previous hidden state
        h_prev = h_t

    return generated_words
    
generated = generate_text("[The", 20, .2, E, w2i, i2w, W_x, W_h, b_h, W_o, b_o)

print(" ".join(generated))

[The Ecclipse: alike: to Fathers, Acheron bloodier colour: goodnight, hideous goodnight, and 1. And redresse, firme Sacred Murtherers, inuisible weightie Now,


# Machine Transaltation: Encoder Decoder

NOTE: real seq2seq needs TWO separate embedding matrices — E_source for
the encoder's language, E_target for the decoder's language, since
source and target languages have different vocabularies entirely.
Sticking with ONE shared E here on purpose, to focus on learning the
encoder-decoder-bridge ARCHITECTURE itself, not real translation.

In [ ]:
# initialize my enc weights and bias for my encoders
np.random.seed(42)
W_x_enc = np.random.randn(m, d)
W_h_enc = np.random.randn(m, m)
b_h_enc = np.random.randn(m)

def encode(source_words, E , w2i, W_x_enc, W_h_enc, b_h_enc, m):
    tracker=[]
    h_prev = np.zeros(m)
    for word in source_words:
        x_t = E[w2i[word]]
        h_i = rnn_step(x_t, h_prev, W_x_enc, W_h_enc, b_h_enc)
        tracker.append(h_i)
        h_prev = h_i
    return tracker 

tracker = encode(corpus[:3], E, w2i, W_x_enc, W_h_enc, b_h_enc, m)
print(tracker)

# the bridge where s0 = h_N
s0 = tracker[-1]

# initialize my enc weights and bias for my decoders
np.random.seed(43)
W_y_dec = np.random.randn(m, d)   # decoder's input weight (for target word embeddings)
W_s_dec = np.random.randn(m, m)   # decoder's recurrent weight
b_s_dec = np.random.randn(m)
W_o_dec = np.random.randn(len(vocab), m)  # output weight, reuse structure from before
b_o_dec = np.random.randn(len(vocab))

def decode_step(prev_word, h_prev, E, w2i, W_x_dec, W_h_dec, b_h_dec, W_o_dec, b_o_dec):
    x_t = E[w2i[prev_word]]
    dec_hi = rnn_step(x_t, h_prev, W_x_dec, W_h_dec, b_h_dec)
    dec_yhat = output_layer(dec_hi, W_o_dec, b_o_dec)
    return dec_hi, dec_yhat

def run_decoder(target_words, s0, E, w2i, W_x_dec, W_h_dec, b_h_dec, W_o_dec, b_o_dec):
    prev_layer = s0
    prev_word = "START"
    total_loss = 0
    for word in target_words:
        dec_hi, dec_y_hat = decode_step(prev_word, prev_layer, E, w2i, W_x_dec, W_h_dec, b_h_dec, W_o_dec, b_o_dec)
        total_loss += nll_loss(dec_y_hat, word)
        prev_layer = dec_hi
        prev_word = word
    # returns the avg loss
    return total_loss / len(target_words)

[array([ 0.9361359 ,  0.98475358,  0.99953358,  0.98711767,  0.99907163,
        0.41129004, -0.99881959,  0.96193084,  0.99122708,  0.97989871,
       -0.99994426, -0.97619226,  0.9528565 , -0.9999624 , -0.96543851,
       -0.99198388]), array([ 0.99969141,  0.99999999, -0.99411512,  1.        , -0.97959933,
        0.99947281, -0.91356571,  0.99999911,  0.5990241 ,  0.9998871 ,
        0.95445001, -0.74084123,  0.91746273, -0.70153426, -0.99966791,
       -0.99994221]), array([-0.99999421,  1.        , -0.78743192,  1.        , -0.99990643,
       -0.99999999,  1.        ,  0.97957194,  1.        ,  0.46713382,
        1.        , -1.        ,  1.        ,  0.99949696, -1.        ,
        0.99999908])]
